In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RANDOMSTATE = 42

# import keras
# from keras import layers


# split into training and testing data 
# standardScaler
# PCA 

# precision
# recall
# Pre-Rec curv 

# confusion matrix 

# f-score 
# ROC AUC

In [2]:
df = pd.read_csv("C:/Users/andsa/Documents/ML/MLing/dataset/sensor.csv")

print(df['machine_status'].value_counts())
df["timestamp"] = pd.to_datetime(df["timestamp"])
# df.drop(columns=["num", "machine_status"], axis=1,  inplace=True)
df.drop(columns=["num"], axis=1,  inplace=True)
df.drop(columns=["sensor_15", "sensor_50", "sensor_51"], axis=1,  inplace=True)
df.columns.str.strip()
df.fillna(method='ffill', inplace=True)

# dropping columns with high linear correlation 
df.drop(columns=["sensor_19", "sensor_18", "sensor_21", "sensor_23", "sensor_24"], axis=1,  inplace=True)

NORMAL        205836
RECOVERING     14477
BROKEN             7
Name: machine_status, dtype: int64


In [3]:
# assigning train to the first 60%, and test to the remaining 40%
last_40percent = round(len(df) - len(df)*.4)
train = df.iloc[ :last_40percent, :]
test = df.iloc[last_40percent: ,:]

X_cols = train.iloc[ : , 1:-1].columns
X_train = train.iloc[:,1:-1].copy()
y_train = train['machine_status'] # suspicious 


X_cols = test.iloc[ : , 1:-1].columns
X_test = test.iloc[:,1:-1].copy()
y_test = test['machine_status']

In [4]:
# scaling and reducing complexity
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components = 0.95, svd_solver = 'full', random_state = RANDOMSTATE )
X_train_PCA = pca.fit_transform(X_train_scaled)
X_test_PCA = pca.transform(X_test_scaled)

print(X_train_PCA.shape)

(132192, 20)


In [5]:
time_data = df['timestamp']

def measurments_per_day(data):
    measurements_per_day = data.dt.date.value_counts().sort_index()
    day_1 = measurements_per_day[0]
    wrong = 0

    for measurments in measurements_per_day:
        if measurments != day_1:
            wrong+= 1
    
    if wrong == 0:
        print(f'All days have the same ammount of measurments: {day_1} ')
    else:
        print(f'All days do not have the same ammount of measurments')

measurments_per_day(time_data)
num_days = time_data.dt.date.nunique()
print(f"Number of days: {num_days}")

All days have the same ammount of measurments: 1440 
Number of days: 153


In [8]:
from sklearn.mixture import BayesianGaussianMixture
import numpy as np

# Ensure you have your X_train_PCA properly defined here
# X_train_PCA = ...

# Create and fit the Bayesian Gaussian Mixture model
# bgm = BayesianGaussianMixture(n_components=10, n_init=10, random_state=42)
bgm = BayesianGaussianMixture(n_components=4, covariance_type='diag', weight_concentration_prior_type='dirichlet_process', weight_concentration_prior=0.1, init_params='kmeans', random_state=RANDOMSTATE)
bgm.fit(X_train_PCA)

# Check the learned weights
print("Component weights:", bgm.weights_.round(2))

# Calculate the log likelihood for each sample in the training data
log_likelihood = bgm.score_samples(X_train_PCA)

# Define a threshold for anomaly detection based on log-likelihood
# For example, you could use a quantile to identify outliers
threshold = np.percentile(log_likelihood, 10)  # Define a threshold

# Mark anomalies as those with log-likelihood below the threshold
anomalies = X_train_PCA[log_likelihood < threshold]

print("Number of anomalies detected:", len(anomalies))


y_pred = bgm.predict(X_test_PCA)
y_true = pd.DataFrame(y_test, columns=['machine_status'])

Component weights: [0.63 0.03 0.12 0.22]
Number of anomalies detected: 13220


In [9]:
from sklearn.metrics import accuracy_score

acc_bay_gauss = accuracy_score(y_true, y_pred)
f"{acc_bay_gauss*100:.1f}%"

'0.0%'

In [ ]:
# TIME_STEPS = 650  # Adjusted from 500

# def create_sequence_generator(values, time_steps=TIME_STEPS):
#     for i in range(len(values) - time_steps + 1):
#         yield values[i : (i + time_steps)]

# # Convert generator to iterable (without excessive RAM usage)
# x_train_gen = create_sequence_generator(X_train_PCA, TIME_STEPS)
# x_test_gen = create_sequence_generator(X_test_PCA, TIME_STEPS)

# # Example: Process one batch at a time (use in training loop)
# for batch in x_train_gen:
#     print(batch.shape)  # Just to verify output
#     break  # Stop after one batch (otherwise, it runs indefinitely)

# x_train = np.array([X_train_PCA[i : i + TIME_STEPS] for i in range(len(X_train_PCA) - TIME_STEPS + 1)])
# # x_test = np.array([X_test_PCA[i : i + TIME_STEPS] for i in range(len(X_test_PCA) - TIME_STEPS + 1)])


In [ ]:
# print(x_train.shape)

In [ ]:
# from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# # Hyperparameters
# BATCH_SIZE = 256
# LEARNING_RATE = 1e-3  # Start higher and let ReduceLROnPlateau adjust it

# # Model Definition
# model = keras.Sequential([
#     layers.Input(shape=(TIME_STEPS, x_train.shape[2])),

#     # Encoder
#     layers.Conv1D(filters=32, kernel_size=7, padding="same", strides=2, activation="relu"),
#     layers.BatchNormalization(),
#     layers.Dropout(rate=0.2),

#     layers.Conv1D(filters=64, kernel_size=5, padding="same", strides=2, activation="relu"),
#     layers.BatchNormalization(),

#     layers.Conv1D(filters=128, kernel_size=3, padding="same", strides=2, activation="relu"),
#     layers.BatchNormalization(),

#     # Decoder (Use UpSampling1D + Conv1D instead of Conv1DTranspose)
#     layers.UpSampling1D(size=2),
#     layers.Conv1D(filters=64, kernel_size=3, padding="same", activation="relu"),
#     layers.BatchNormalization(),

#     layers.UpSampling1D(size=2),
#     layers.Conv1D(filters=32, kernel_size=5, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.Dropout(rate=0.2),

#     layers.UpSampling1D(size=2),
#     layers.Conv1D(filters=x_train.shape[2], kernel_size=7, padding="same", activation="relu"),

#     # Final output layer
#     layers.Conv1D(filters=x_train.shape[2], kernel_size=1, padding="same"),

#     # 🛠 Fix: Cropping to match original size if necessary
#     layers.Cropping1D(cropping=(3, 0))  # Adjust cropping value as needed
# ])

# # Learning Rate Decay
# lr_scheduler = ReduceLROnPlateau(
#     monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, mode="min"
# )

# # Compile Model
# optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
# model.compile(optimizer=optimizer, loss="mse")

# # Print model summary
# model.summary()


In [ ]:
# # Callbacks: EarlyStopping + ReduceLROnPlateau
# callbacks = [
#     EarlyStopping(monitor="val_loss", patience=5, mode="min"),
#     lr_scheduler
# ]

# # Train the model
# history = model.fit(
#     x_train, x_train,  # Autoencoder: input = output
#     epochs=50,
#     batch_size=BATCH_SIZE,
#     validation_split=0.1,
#     callbacks=callbacks
# )

In [ ]:
# TIME_STEPS = 1400  # Now using full-day sequences

# # Function to create sequences
# def create_sequences(values, time_steps=TIME_STEPS):
#     output = []
#     for i in range(len(values) - time_steps + 1):
#         output.append(values[i : (i + time_steps)])
#     return np.stack(output)

# # Creating sequences for full-day analysis
# x_train = create_sequences(X_train_PCA, TIME_STEPS)
# x_test = create_sequences(X_test_PCA, TIME_STEPS)

# print("Training input shape: ", x_train.shape)
# print("Test input shape: ", x_test.shape)

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA

# TIME_STEPS = 1400  # Now using full-day sequences

# # Function to create sequences
# def create_sequences(values, time_steps=TIME_STEPS):
#     output = []
#     for i in range(len(values) - time_steps + 1):
#         output.append(values[i : (i + time_steps)])
#     return np.stack(output)

# # Load and preprocess data
# df = pd.read_csv("C:/Users/andsa/Documents/ML/MLing/dataset/sensor.csv")

# df["timestamp"] = pd.to_datetime(df["timestamp"])
# df.drop(columns=["num"], axis=1, inplace=True)
# df.drop(columns=["sensor_15", "sensor_50", "sensor_51"], axis=1, inplace=True)
# df.fillna(method='ffill', inplace=True)

# # Splitting into train and test sets (60% train, 40% test)
# last_40percent = round(len(df) - len(df)*.4)
# train = df.iloc[:last_40percent, :]
# test = df.iloc[last_40percent:, :]

# # Selecting features (excluding timestamp and labels)
# X_train = train.iloc[:, 1:-1].copy()
# X_test = test.iloc[:, 1:-1].copy()

# # Standardizing the data
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Applying PCA
# pca = PCA(n_components=0.95, svd_solver='full', random_state=42)
# X_train_PCA = pca.fit_transform(X_train_scaled)
# X_test_PCA = pca.transform(X_test_scaled)

# # Creating sequences for full-day analysis
# x_train = create_sequences(X_train_PCA, TIME_STEPS)
# x_test = create_sequences(X_test_PCA, TIME_STEPS)

# print("Training input shape: ", x_train.shape)
# print("Test input shape: ", x_test.shape)


In [ ]:
# df = pd.read_csv("C:/Users/andsa/Documents/ML/MLing/dataset/sensor.csv")
# # df["timestamp"] = pd.to_datetime(df["timestamp"])
# # df.info()
# # df.drop(columns=["num", "timestamp", "machine_status"], axis=1,  inplace=True)

# df.drop(columns=["num", "timestamp"], axis=1,  inplace=True)
# df.drop(columns=["sensor_15", "sensor_50", "sensor_51"], axis=1,  inplace=True)
# df.columns.str.strip()
# df.fillna(method='ffill', inplace=True)



In [ ]:
# # for items in df["machine_status"]:
# #     if items == "NORMAL":
# #         print(items)

# def find_occurance(df, column, token):
#     num = 0
#     for status in df[column]:
#         if status == token:
#             num += 1
#     return num
# # num_of_broken = find_occurance(df, 'machine_status', 'BROKEN')
# # print(f'The number of BROKEN occurances are: {num_of_broken}')


# def return_columns_with_lots_of_NAN(df, limit):
#     columns_to_remove = ''

#     for column in df:
#         column_length = len(df[column])
#         nan = df[column].isna().sum()

#         fraction = nan / column_length

#         if fraction >= limit:
#             columns_to_remove += (column + ', ')
#     return columns_to_remove 
# # nan_string = return_columns_with_lots_of_NAN(df, 0.05)
# # print(f'Columns with more than the limit of NAN values: {nan_string}')

# # dropping columns with high linear correlation 
# df.drop(columns=["sensor_19", "sensor_18", "sensor_21", "sensor_23", "sensor_24"], axis=1,  inplace=True)

# # https://www.geeksforgeeks.org/principal-component-analysis-pca/

In [ ]:

# split into training and testing data 
# standardScaler
# PCA 

# precision
# recall
# Pre-Rec curv 

# f-score 
# ROC AUC

In [ ]:
# df.drop(columns=["timestamp"], axis=1,  inplace=True)
# # assigning the X and y values 
# X = df.drop(['machine_status'], axis = 1)
# y = df[['machine_status']]

# # importing a splitter for the data 
# from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.6, random_state=RANDOMSTATE)
# # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = pd.qcut(y, q = 5, duplicates='drop'))

# # Scaling the data 
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

In [ ]:
# print(df.isnull().sum())
# df.head()
# df.describe()

# Select the first 3 columns, excluding "num" and "timestamp"
# columns_to_plot = df.columns[:5]  # Get the first 3 columns

# Loop over the selected columns
# for column in columns_to_plot:
#     if column == "num" or column == "timestamp":
#         continue  # Skip "num" and "timestamp"
    
#     plt.figure(figsize=(22, 1), dpi=600)
#     plt.title(column)
#     plt.plot(df["num"], df[column])  # Plot against the "num" column
#     plt.show()

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.preprocessing import StandardScaler
# from keras.models import Sequential
# from keras.layers import Dense
# from sklearn.metrics import mean_squared_error
# import seaborn as sns

# # Step 1: Load the data
# df = pd.read_csv("C:/Users/andsa/Documents/ML/MLing/dataset/sensor.csv")

# df.drop(columns=["machine_status", "sensor_15"], axis=1,  inplace=True)
# df.fillna(method='ffill', inplace=True)
# df.columns = df.columns.str.strip()

# # Convert 'timestamp' to datetime and remove it
# df["timestamp"] = pd.to_datetime(df["timestamp"])
# df = df.drop(columns=["timestamp"])

# # Standardize the data
# scaler = StandardScaler()
# data_scaled = scaler.fit_transform(df)

# # Step 2: Build the Autoencoder Model
# # Define the autoencoder model architecture
# autoencoder = Sequential()
# autoencoder.add(Dense(64, input_dim=data_scaled.shape[1], activation='relu'))
# autoencoder.add(Dense(32, activation='relu'))
# autoencoder.add(Dense(16, activation='relu'))
# autoencoder.add(Dense(32, activation='relu'))
# autoencoder.add(Dense(64, activation='relu'))
# autoencoder.add(Dense(data_scaled.shape[1], activation='sigmoid'))  # Output layer

# # Compile the model
# autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# # Step 3: Train the Autoencoder
# # Train the autoencoder on the normal data (since it's unsupervised, we don't need labels)
# autoencoder.fit(data_scaled, data_scaled, epochs=50, batch_size=256, shuffle=True)

# # Step 4: Use the trained model to detect anomalies
# # Compute the reconstruction error for each data point
# reconstructed = autoencoder.predict(data_scaled)
# reconstruction_error = mean_squared_error(data_scaled, reconstructed, multioutput='raw_values')

# # Set a threshold for anomaly detection (e.g., based on the distribution of the error)
# threshold = np.percentile(reconstruction_error, 95)  # 95th percentile as the threshold for anomaly

# # Mark anomalies where the reconstruction error exceeds the threshold
# anomalies = reconstruction_error > threshold

# # Step 5: Visualize the results
# plt.figure(figsize=(10,6))
# plt.plot(df.index, reconstruction_error, label='Reconstruction Error')
# plt.axhline(y=threshold, color='r', linestyle='--', label='Anomaly Threshold')
# plt.scatter(df.index[anomalies], reconstruction_error[anomalies], color='r', label='Anomalies')
# plt.title('Reconstruction Error and Anomalies')
# plt.xlabel('Index')
# plt.ylabel('Reconstruction Error')
# plt.legend()
# plt.show()

# # Step 6: Output the anomalies
# df['anomaly'] = anomalies
# print(df[df['anomaly'] == True])  # Display rows identified as anomalies
